# TD/TME8: SQL2 – Jointures Naturelles, Externes, Sous-requêtes

On utilise  [DuckDB](https://duckdb.org). Voir la [documentation DuckDB SQL](https://duckdb.org/docs/sql/introduction.html)

In [1]:
pip install duckdb

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Démarrer le service DuckDB
import duckdb

db = duckdb.connect(':memory:')


# vérifier que le service fonctionne
r = db.sql("SELECT 'hello' as col")
display(r)

┌─────────┐
│   col   │
│ varchar │
├─────────┤
│ hello   │
└─────────┘

# Créer les tables et charger les données bd_jo_v2_duck.sql

Ce TD/TME utilise les données contenues dans le fichier **bd_jo_v2_duck.sql**

**Attention**: vous pouvez cliquer sur les 3 lignes dans la fenêtre de gauche pour d'afficher les différentes sections du notebook   

In [3]:
with open('bd-jo-v2_duck.sql', 'r',encoding='utf-8') as file:
    data = file.read()
db.execute(data)    

**Consulter la BD**

La commande `SHOW TABLES` retourne le nom des tables.

Note: la fonction *.df()* ajoutée à la fin de l'expression sert à importer le résultat de la requête dans un dataframe pandas afin d'avoir un rendu "ergonomique".

In [5]:
db.execute("SHOW TABLES").df()

,name
0,Athlete
1,AthletesEquipe
2,Epreuve
3,Equipe
4,Pays
5,RangEquipe
6,RangIndividuel
7,Sport


In [6]:
show_table('Athlete')

NameError: name 'show_table' is not defined

La commande `DESCRIBE` retourne une description des attributs d'une table.

Exemple affichant les attributs de la table *Athlete* :

In [ ]:
db.execute("DESCRIBE Athlete").df()

,column_name,column_type,null,key,default,extra
0,aid,INTEGER,NO,PRI,None,None
1,nomAth,VARCHAR,YES,NaN,None,None
2,prenomAth,VARCHAR,YES,NaN,None,None
3,dateNaissance,DATE,YES,NaN,None,None
4,codePays,VARCHAR,YES,NaN,None,None


La clause  `LIMIT N` est ajoutée à la fin d'une requête pour calculer seulement les N premiers tuples du résultat.

Exemple pour afficher 10 tuples de la table *Athlete*:

In [4]:
db.execute("SELECT * FROM Athlete LIMIT 10").df()

,aid,nomAth,prenomAth,dateNaissance,codePays
0,1,BJOERNDALEN,Ole Einar,1974-01-27,NOR
1,2,BJOERGEN,Marit,1980-03-21,NOR
2,3,AN,Victor,1985-11-23,RUS
3,4,PECHSTEIN,Claudia,1972-02-22,GER
4,5,WÜST,Ireen,1986-04-01,NED
5,6,SVENDSEN,Emil Hegle,1985-07-12,NOR
6,7,AMMANN,Simon,1981-06-25,SUI
7,8,KRAMER,Sven,1986-04-23,NED
8,9,SABLIKOVA,Martina,1987-05-27,CZE
9,10,HAMELIN,Charles,1984-04-14,CAN


In [ ]:
#Afficher toutels les lignes de la table Athlète
db.execute("select * from athlete").df()

,aid,nomAth,prenomAth,dateNaissance,codePays
0,1,BJOERNDALEN,Ole Einar,1974-01-27,NOR
1,2,BJOERGEN,Marit,1980-03-21,NOR
2,3,AN,Victor,1985-11-23,RUS
3,4,PECHSTEIN,Claudia,1972-02-22,GER
4,5,WÜST,Ireen,1986-04-01,NED
...,...,...,...,...,...
2426,2427,NARUSE,Nobu,NaT,JPN
2427,2428,YOSHIDA,Keishin,NaT,JPN
2428,2429,MIYAZAWA,Hiroyuki,NaT,JPN
2429,2430,LENTING,Akira,NaT,JPN


# TD8: SQL2 – Jointures Naturelles, Externes, Sous-requêtes

On considère le schéma de la base JEUXOLYMPIQUES2014 qui décrit les athlètes et leurs résultats aux
épreuves des sports des Jeux Olympiques d'Hiver Sotchi 2014 :

**PAYS** ( <u>CODEPAYS</u>, NOMP)<br/>
Ex. ('FRA', 'France')<br/>
**SPORT** ( <u>SID</u>, NOMSP)<br/>
Ex. (1, 'Biathlon')<br/>
**EPREUVE** ( <u>EPID</u>, SID*, NOMEP, CATÉGORIE, DATEDEBUT, DATEFIN)<br/>
Ex. (10, 1, 'relais 4x7,5km', 'Hommes', 22/02/2014, 22/02/2014)<br/>
**ATHLETE** ( <u>AID</u>, NOMATH, PRENOMATH, DATENAISSANCE, CODEPAYS*)<br/>
Ex. (1000, 'SOBOLEV', 'Alexey', NULL, 'RUS')<br/>
**EQUIPE** ( <u>EQID</u>, CODEPAYS*)<br/>
Ex. (30, 'SUI')<br/>
**ATHLETESEQUIPE** ( <u>EQID*, AID*</u>)<br/>
Ex. (30, 796) : L'athlète (aid=796) a participé à l'équipe (eqid=30)<br/>
**RANGINDIVIDUEL** ( <u>EPID*, AID*</u>, RANG)<br/>
Ex. (15, 61, 1) : L'athlète (aid=61) a gagné la médaille d'or (rang=1) de l'épreuve (epid=15)<br/>
**RANGEQUIPE** ( <u>EPID*, EQID*</u>, RANG)<br/>
Ex. (10, 30, 14) : L'équipe (eqid=30) a été classée 14e à l'épreuve (epid=10)<br/>


Les attributs qui forment la clé primaire de chaque relation sont soulignés. Les clés étrangères sont
signalées avec une *. Les attributs aid, epid, eqid et sid correspondent aux identifiants des athlètes,
épreuves, équipes et sports et sont utilisés à la fois comme clé primaire ou comme référence (clé
étrangère) vers la relation correspondante.
La relation **PAYS** contient le code et le nom de tous les pays, même si ils n'ont pas participé aux
Jeux Olympiques. Les sports (n-uplets de la relation **SPORT**) sont un ensemble d'épreuves (n-uplets
de la relation **EPREUVE**). Pour chaque épreuve on connaît son nom et les date de début et fin de
l'épreuve. Les épreuves peuvent être individuelles ou par équipe. Dans le premier cas, la
participation des athlètes (n-uplets de la relation ATHLETE) est stocké dans la table
**RANGINDIVIDUEL** qui contient en plus le rang qu'ils ont obtenu (1 pour la médaille d'or). Pour les
épreuves par équipe les résultats sont stockés dans la relation **RANGEQUIPE**, alors que l'information
sur le pays de chaque équipe et ses participants et stocké dans les relations **EQUIPE** et
**ATHLETESEQUIPE**. Dans les relations **RANGINDIVIDUEL** et **RANGEQUIPE** l'attribut rang est égal à
null si l'athlète ou l'équipe a été disqualifié.


## Requêtes

### Jointures internes  « INNER JOIN »

#### **Q1**. 
Les noms et les prénoms des athlètes français (nom pays = 'France') (104 lignes).

In [ ]:
query="""
select  a.nomath, a.prenomath
from athlete a inner join pays p on a.codepays = p.codepays 
where p.nomp ='France'
"""

db.execute(query).df()

,nomAth,prenomAth
0,FOURCADE,Martin
1,LAMY CHAPPUIS,Jason
2,VAULTIER,Pierre
3,CHAPUIS,Jean Frederic
4,BRUNET,Marie Laure
...,...,...
99,Daniel Jean-Paul RICARD,Vincent
100,BOURZAT,Fabian
101,PECHALAT,Nathalie
102,CARRON,Pernelle


#### **Q2** . 
Les épreuves (sport, nom d'épreuve) triées par nom de sport, puis par nom d'épreuve dans l'ordre inverse du dictionnaire (66 lignes).

In [ ]:
query="""
select distinct s.nomsp, e.nomep
from sport s inner join epreuve e on e.sid = s.sid
order by s.nomsp , e.nomep desc
"""

db.execute(query).df()

,nomSp,nomEp
0,Biathlon,"relais 4x7,5km"
1,Biathlon,relais 4x6km
2,Biathlon,Relais mix
3,Biathlon,"7,5km"
4,Biathlon,20km
...,...,...
61,Surf des neiges,Snowboard cross
62,Surf des neiges,Slopestyle
63,Surf des neiges,Slalom parallèle
64,Surf des neiges,Slalom géant parallèle


#### **Q3** . 
Les homonymes (les nom de familles portés par deux athlètes ou plus) (141 lignes)

In [ ]:
query="""
select distinct a.nomath
from athlete a inner join athlete a2 on a.nomath = a2.nomath
where a.aid <> a2.aid
"""

db.execute(query).df()

,nomAth
0,HAMELIN
1,COLOGNA
2,DAVIS
3,FOURCADE
4,SCHOCH
...,...
136,POULSEN
137,Marty
138,Jokinen
139,LINGER


#### **Q4**. 
Les athlètes ayant participé à (au moins) deux épreuves individuelles (706 lignes).

In [ ]:
query="""
select distinct a.*
from 
athlete a 
inner join rangindividuel r on r.aid = a.aid 
inner join rangindividuel r2 on r2.aid = a.aid 
where r.epid <> r2.epid

"""
 
db.execute(query).df()

,aid,nomAth,prenomAth,dateNaissance,codePays
0,2,BJOERGEN,Marit,1980-03-21,NOR
1,3,AN,Victor,1985-11-23,RUS
2,4,PECHSTEIN,Claudia,1972-02-22,GER
3,5,WÜST,Ireen,1986-04-01,NED
4,6,SVENDSEN,Emil Hegle,1985-07-12,NOR
...,...,...,...,...,...
701,1739,PANTOV,Anton,NaT,KAZ
702,1741,USANOVA,Darya,NaT,KAZ
703,1744,ARWIDSON,Tobias,NaT,SWE
704,1762,GASPARIN,Elisa,NaT,SUI


### Jointures naturelles « NATURAL JOIN » 

#### **Q5**. 
Les noms et les prénoms des athlètes français (nom pays = 'France') (104 lignes).

In [ ]:
query="""

select nomath, prenomath
from athlete natural join pays 
where nomp = 'France'
"""

db.execute(query).df()

,nomAth,prenomAth
0,FOURCADE,Martin
1,LAMY CHAPPUIS,Jason
2,VAULTIER,Pierre
3,CHAPUIS,Jean Frederic
4,BRUNET,Marie Laure
...,...,...
99,Daniel Jean-Paul RICARD,Vincent
100,BOURZAT,Fabian
101,PECHALAT,Nathalie
102,CARRON,Pernelle


#### **Q6** . 
Les épreuves (sport, nom d'épreuve) triées par nom de sport, puis par nom d'épreuve dans l'ordre inverse du dictionnaire (66 lignes).

In [ ]:
query="""
select distinct s.nomsp, e.nomep
from sport s natural join epreuve e
order by s.nomsp , e.nomep desC
"""

db.execute(query).df()

,nomSp,nomEp
0,Biathlon,"relais 4x7,5km"
1,Biathlon,relais 4x6km
2,Biathlon,Relais mix
3,Biathlon,"7,5km"
4,Biathlon,20km
...,...,...
61,Surf des neiges,Snowboard cross
62,Surf des neiges,Slopestyle
63,Surf des neiges,Slalom parallèle
64,Surf des neiges,Slalom géant parallèle


#### **Q7** . 
Les sports (identifiant et nom) et les épreuves (identifiant et nom) en équipe (25 lignes).

In [ ]:
query="""
select distinct s.nomsp , s.sid , e.epid , e.nomep
from sport s natural join epreuve e natural join  rangequipe r 

"""

db.execute(query).df()

,nomSp,sid,epid,nomEp
0,Biathlon,1,5,relais 4x6km
1,Biathlon,1,10,"relais 4x7,5km"
2,Biathlon,1,11,Relais mix
3,Bobsleigh,2,12,bob à deux
4,Bobsleigh,2,13,bob à deux
5,Bobsleigh,2,14,bob à quatres
6,Combiné nordique,3,17,Par équipes
7,Curling,4,19,curling
8,Hockey sur glace,5,20,hockey sur glace
9,Hockey sur glace,5,21,hockey sur glace


#### **Q8** . 
Les noms et les prénoms des athlètes qui ont gagné au moins une médaille en équipe au sport 'Biathlon' (34 lignes).

In [ ]:
query="""
select distinct a.nomath ,a.prenomath
from athlete a natural join athletesequipe aq 

natural join                 rangequipe rq 
natural join                 epreuve e 
natural join                 sport s 

where s.nomsp = 'Biathlon' and rq.rang <= 3

"""

db.execute(query).df()

,nomAth,prenomAth
0,SVENDSEN,Emil Hegle
1,ZAITSEVA,Olga
2,BERGER,Tora
3,USTYUGOV,Evgeny
4,SEMERENKO,Vita
5,VOLKOV,Alexey
6,DZHYMA,Juliya
7,PIDHRUSHNA,Olena
8,MALYSHKO,Dmitry
9,SUMANN,Christoph


### Jointures externes « LEFT|RIGHT|FULL OUTER JOIN » 

#### **Q9**. 
Pour chaque pays, le nombre de médailles en épreuve individuelle gagnées dans l'ordre décroissant des médailles gagnées (on veut aussi les pays sans médailles) (206 lignes)

In [ ]:
query="""
select p.codepays , count(r.rang)
from pays p left join athlete a on p.codepays = a.codepays
left join rangindividuel r on a.aid = r.aid 
group by p.codepays
order by count(r.rang) desc

"""

db.execute(query).df()

,codePays,count(r.rang)
0,USA,173
1,RUS,152
2,CAN,150
3,NOR,134
4,GER,133
...,...,...
201,JOR,0
202,KEN,0
203,LAO,0
204,LCA,0


#### **Q10**. 
Tous les numéros d'équipes avec les numéros de leurs membres (NULL si les membres de l'équipe sont inconnus) (1108 lignes)

In [ ]:
query="""
select e.eqid , aq.aid 
from equipe e left join athletesequipe aq on e.eqid = aq.eqid

"""

db.execute(query).df()

,eqid,aid
0,1,107
1,1,120
2,1,92
3,1,121
4,2,21
...,...,...
1103,231,<NA>
1104,233,<NA>
1105,232,<NA>
1106,234,<NA>


In [ ]:
query="""
select *
from athlete
"""

db.execute(query).df()

,aid,nomAth,prenomAth,dateNaissance,codePays
0,1,BJOERNDALEN,Ole Einar,1974-01-27,NOR
1,2,BJOERGEN,Marit,1980-03-21,NOR
2,3,AN,Victor,1985-11-23,RUS
3,4,PECHSTEIN,Claudia,1972-02-22,GER
4,5,WÜST,Ireen,1986-04-01,NED
...,...,...,...,...,...
2426,2427,NARUSE,Nobu,NaT,JPN
2427,2428,YOSHIDA,Keishin,NaT,JPN
2428,2429,MIYAZAWA,Hiroyuki,NaT,JPN
2429,2430,LENTING,Akira,NaT,JPN


#### **Q11**. 
Les numéros des équipes dont on ne connaît pas les membres (utilisez la jointure externe) (13 lignes).

In [ ]:
query="""
select e.eqid 
from equipe e left join athletesequipe aq on e.eqid = aq.eqid 
where aq.aid is null
"""

db.execute(query).df()

,eqid
0,163
1,229
2,235
3,160
4,162
5,159
6,167
7,230
8,231
9,233


#### **Q12**. 
Tous les noms d'athlètes avec les numéros de leurs équipes (NULL si l'athlète n'appartient à aucune équipe) (2579 lignes).

In [ ]:
query="""
select a.nomath
from athlete a left outer join athletesequipe aq on a.aid = aq.aid 
"""

db.execute(query).df()

,nomAth
0,BJOERNDALEN
1,BJOERGEN
2,AN
3,WÜST
4,SVENDSEN
...,...
2574,POREBA
2575,ZAKHARKIV
2576,OBOLONCHYK
2577,PARK


#### **Q13**.
Les epreuves (noms du sport et de l'épreuve) avec un attribut pour chaque catégorie (Hommes, Femmes, Mixte). Par exemple, la requête retourne les nuplets ('Bobsleigh', 'bob à deux', 'Hommes',  'Femmes' , NULL) et ('Luge', 'Double', NULL, NULL, 'Mixte') ? (66 lignes)

In [ ]:
query="""
select distinct e.nomep , s.nomsp, eh.categorie as Homme, ef.categorie as Femme , em.categorie as Mixte
from epreuve e join sport s on s.sid = e.sid 
left join epreuve eh on eh.sid = e.sid and eh.nomep = e.nomep and eh.categorie = 'Hommes'
left join epreuve ef on ef.sid = e.sid and ef.nomep = e.nomep and ef.categorie = 'Femmes'
left join epreuve em on em.sid = e.sid and em.nomep = e.nomep and em.categorie = 'Mixte'
"""

db.execute(query).df()

,nomEp,nomSp,Homme,Femme,Mixte
0,bob à deux,Bobsleigh,Hommes,Femmes,NaN
1,curling,Curling,Hommes,Femmes,NaN
2,1000m,Patinage de vitesse,Hommes,Femmes,NaN
3,2x500m,Patinage de vitesse,Hommes,Femmes,NaN
4,500m,Patinage de vitesse sur piste courte,Hommes,Femmes,NaN
...,...,...,...,...,...
61,50km,Ski de fond,Hommes,NaN,NaN
62,15km,Biathlon,NaN,Femmes,NaN
63,3000m,Patinage de vitesse,NaN,Femmes,NaN
64,10km,Ski de fond,NaN,Femmes,NaN


### Requêtes dans la clause FROM et top-k

#### **Q14**. 
Le nom et l'age actuel des 10 athlètes les plus jeunes (utiliser *date_sub*). 

In [ ]:
query="""
select a.nomath , year (current_date) - year (a.datenaissance) as AGE

from athlete a
order by a.datenaissance desc
limit 10
"""

db.execute(query).df()

,nomAth,AGE
0,HIRANO,28
1,LIPNITSKAYA,28
2,SHIM,29
3,SOTNIKOVA,30
4,TIANYU,30
5,OSMOND,31
6,MATTEL,31
7,HIRAOKA,31
8,CHEN,31
9,WELLINGER,31


#### **Q15**. 
Le nom et l'age actuel des 10 athlètes les plus âgés. 

In [ ]:
query="""
select a.nomath , year (current_date) - year(a.datenaissance) as AGE
from athlete a
order by a.datenaissance asc
limit 10
"""
db.execute(query).df()

,nomAth,AGE
0,DEMTSCHENKO,55
1,PECHSTEIN,54
2,KASAI,54
3,DI CENTA,54
4,ZOEGGELER,52
5,BJOERNDALEN,52
6,TABATA,52
7,ANDERSON,51
8,SUMANN,50
9,MESOTITSCH,50


#### **Q16**. 
Le nombre moyen d'athlètes par pays (avec group by). Aide : compter le nombre
d’athlètes dans chaque pays (ayant au moins un athlète), puis faire la moyenne. Résultat (1 ligne) : 27,625

In [ ]:
query="""
select avg(t.x) as Moyennn
from (select count (*) as x
      from athlete a 
      group by a.codepays ) t

"""

db.execute(query).df()

,Moyennn
0,27.625


#### **Q17**. 
L’eqid de la ou des équipes qui sont composées du plus d’athlètes pour ces JO.
 Résultats (3 lignes) : 164 ; 165 ; 166 

In [ ]:
query="""

select a.eqid , count (*)
from athletesequipe a
group by a.eqid 
having count(*) = (select  max(t.nb)

                from ( select count (*) as nb
                from athletesequipe aq
                group by aq.eqid ) t
)

"""

db.execute(query).df()

,eqid,count_star()
0,164,25
1,165,25
2,166,25


#### **Q18**.
Le nombre d'épreuves en individuel où il y a eu au moins 100 participants. Résultat (1 ligne ) : 2

In [ ]:
query="""
select count(*)
from ( select distinct r.epid 
    from   rangindividuel r
    group by r.epid 
    having count(*) >= 100 )

"""

db.execute(query).df()

,count_star()
0,2


# TME8: SQL2 – Jointures Naturelles, Externes, Sous-requêtes

Ce TME utilise les données contenues dans le fichier **bd_jo_v2_duck.sql**


## Requêtes

### Jointures naturelles « NATURAL JOIN » et Jointures Internes  «INNER JOIN »  

#### **Q1-a)**. 
Les noms des épreuves individuelles (identifiants et noms) et les noms des sports correspondants (identifiants et noms), avec Natural Join (73 lignes).

In [ ]:
query="""
select distinct e.nomep, e.epid,s.nomsp , s.sid
from rangindividuel r natural join epreuve e natural join sport s
"""

db.execute(query).df()

,nomEp,epid,nomSp,sid
0,"7,5km",4,Biathlon,1
1,10km,6,Biathlon,1
2,"12,5km poursuite",7,Biathlon,1
3,Simple,22,Luge,6
4,Simple,23,Luge,6
...,...,...,...,...
68,Slalom parallèle,91,Surf des neiges,15
69,Slopestyle,92,Surf des neiges,15
70,Snowboard cross,93,Surf des neiges,15
71,Slalom parallèle,96,Surf des neiges,15


#### **Q1-b)**. 
Les noms des épreuves individuelles (identifiants et noms) et les noms des sports correspondants (identifiants et noms), avec Inner Join (73 lignes).

In [ ]:
query="""
select distinct e.epid , e.nomep ,s.sid ,s.nomsp
from sport s 
inner join  epreuve e on e.sid = s.sid 
inner join rangindividuel r on r.epid = e.epid
"""

db.execute(query).df()

,epid,nomEp,sid,nomSp
0,1,10km poursuite,1,Biathlon
1,2,"12,5km départ groupé",1,Biathlon
2,3,15km,1,Biathlon
3,4,"7,5km",1,Biathlon
4,6,10km,1,Biathlon
...,...,...,...,...
68,90,Slalom géant parallèle,15,Surf des neiges
69,91,Slalom parallèle,15,Surf des neiges
70,92,Slopestyle,15,Surf des neiges
71,96,Slalom parallèle,15,Surf des neiges


#### **Q2-a)**.
Les noms des sports auxquels LESSER Erik a participé en individuel avec Natural Join (Résultats 2 lignes: Biathlon, Ski de fond).

In [ ]:
query="""
select distinct s.nomsp
from epreuve e 
natural join sport s 
natural join rangindividuel r
natural join athlete a
where a.nomath = 'LESSER' and a.prenomath = 'Erik'

"""

db.execute(query).df()

,nomSp
0,Biathlon
1,Ski de fond


#### **Q2-b)**.
Les noms des sports auxquels LESSER Erik a participé en individuel avec Inner Join (Résultats 2 lignes: Biathlon, Ski de fond).

In [ ]:
query="""
select distinct s.nomsp
from sport s 
inner join epreuve e on e.sid = s.sid 
inner join rangindividuel r on r.epid = e.epid
inner join athlete a on a.aid = r.aid
where a.nomath = 'LESSER' and a.prenomath = 'Erik'
"""

db.execute(query).df()

,nomSp
0,Biathlon
1,Ski de fond


#### **Q3-a)**. 
Le nom et prénom des athlètes qui ont gagné la médaille d'or dans l'épreuve par équipe 'relais 4x6km' de 'Biathlon' de 'Femmes' avec Natural Join (Résultat : SEMERENKO Vita, SEMERENKO Valj, DZHYMA Juliya, PIDHRUSHNA Olena).

In [ ]:
query="""
select a.nomath , a.prenomath
from athlete a 
natural join athletesequipe aq
natural join rangequipe rq
natural join epreuve e
natural join sport s
where s.nomsp = 'Biathlon' and e.categorie = 'Femmes' and rq.rang = 1 and e.nomep = 'relais 4x6km'

"""

db.execute(query).df()

,nomAth,prenomAth
0,SEMERENKO,Vita
1,SEMERENKO,Valj
2,DZHYMA,Juliya
3,PIDHRUSHNA,Olena


#### **Q3-b)**. 
Le nom et prénom des athlètes qui ont gagné la médaille d'or dans l'épreuve par équipe 'relais 4x6km' de 'Biathlon' de 'Femmes' avec Inner Join (Résultat : SEMERENKO Vita, SEMERENKO Valj, DZHYMA Juliya, PIDHRUSHNA Olena).

In [ ]:
query="""
select a.nomath , a.prenomath
from athlete a 
inner join athletesequipe aq on a.aid = aq.aid 
inner join rangequipe rq on rq.eqid = aq.eqid 
inner join epreuve e on e.epid = rq.epid 
inner join sport s on s.sid = e.sid 
where e.categorie = 'Femmes' and s.nomsp = 'Biathlon' and e.nomep = 'relais 4x6km' and rq.rang =1

"""

db.execute(query).df()

,nomAth,prenomAth
0,SEMERENKO,Vita
1,SEMERENKO,Valj
2,DZHYMA,Juliya
3,PIDHRUSHNA,Olena


#### **Q4**. 
La liste des pays, avec les épreuves (individuels et en équipe) et les rangs obtenus (293 lignes).

In [ ]:
query="""
SELECT a.codePays, e.nomEp, r.rang
FROM rangIndividuel r
JOIN athlete a ON a.aid = r.aid
JOIN epreuve e ON e.epid = r.epid
WHERE r.rang BETWEEN 1 AND 3

UNION ALL

SELECT eq.codePays, e.nomEp, r.rang
FROM rangEquipe r
JOIN equipe eq ON eq.eqid = r.eqid
JOIN epreuve e ON e.epid = r.epid
WHERE r.rang BETWEEN 1 AND 3

"""

db.execute(query).df()

,codePays,nomEp,rang
0,NOR,10km,1
1,NOR,"Skiathlon (7,5km + 7,5km)",1
2,RUS,1500m,3
3,NED,5000m,2
4,NOR,15km départ groupé,1
...,...,...,...
288,GER,Par équipes,1
289,SWE,Relais 4x5km,1
290,NOR,Sprint par équipes,1
291,SWE,Relais 4x10km,1


#### **Q5**. 
Les athlètes qui sont plus jeunes que LESSER Erik, avec Inner join (132 lignes).

In [ ]:
query="""
select a.nomath
from athlete a
inner join athlete l on a.datenaissance > l.datenaissance
and l.nomath = 'LESSER' and l.prenomath = 'Erik'
"""

db.execute(query).df()

,nomAth
0,LOCH
1,ZHOU
2,FOURCADE
3,PARK
4,SANG-HWA
...,...
127,KRISTOFFERSEN
128,WINDISCH
129,ROLLAND
130,GOEPPER


### Jointures Externes  « LEFT|RIGHT|FULL OUTER JOIN »   

#### **Q6**. 
Les noms de tous les athlètes avec les numéros des équipes (le numéro est NULL si l'athlète n'appartient à aucune équipe et le nom est NULL si les membres d'une équipe sont inconnus) (2592 lignes).


In [ ]:
query="""
SELECT a.nomAth, ae.eqid
FROM athletesEquipe ae
FULL OUTER JOIN athlete a
ON ae.aid = a.aid
"""
# ! ! ! 
db.execute(query).df()

,nomAth,eqid
0,BJOERNDALEN,36
1,BJOERGEN,275
2,AN,244
3,WÜST,236
4,SVENDSEN,36
...,...,...
2574,POREBA,182
2575,ZAKHARKIV,184
2576,OBOLONCHYK,184
2577,PARK,185


###  Sous-requêtes dans FROM/SELECT  

#### **Q7**. 
Le tableau complet des résultats (sport, épreuve, catégorie, athlète, rang) trié par sport, épreuve, catégorie et rang (3780 lignes).

In [36]:
query="""
select s.nomsp , e.nomep , e.categorie, a.nomath , r.rang
from sport s 
natural join epreuve e 
natural join rangindividuel r 
natural join athlete a

union all 

select s.nomsp , e.nomep , e.categorie, a.nomath , r.rang
from sport s 
natural join epreuve e 
natural join rangequipe r 
natural join athletesequipe aq 
natural join athlete a


ORDER BY nomSp, nomEp, categorie, rang
"""
#  better to use inner join +  on
db.execute(query).df()

,nomSp,nomEp,categorie,nomAth,rang
0,Biathlon,10km,Hommes,BJOERNDALEN,1
1,Biathlon,10km,Hommes,LANDERTINGER,2
2,Biathlon,10km,Hommes,SOUKUP,3
3,Biathlon,10km,Hommes,SHIPULIN,4
4,Biathlon,10km,Hommes,LEGUELLEC,5
...,...,...,...,...,...
3775,Surf des neiges,Snowboard cross,Hommes,DE LE RUE,4
3776,Surf des neiges,Snowboard cross,Hommes,SIVERTZEN,5
3777,Surf des neiges,Snowboard cross,Hommes,MATTEOTTI,6
3778,Surf des neiges,Snowboard cross,Hommes,EGUIBAR,7


#### **Q8**. 
Pour chaque pays, le nom et le nombre de médailles gagnées (en individuel ou en équipe). Conseil : utilisez la requête 1.4 comme sous-requête dans  FROM (25 lignes)

In [56]:
query="""
SELECT codePays, COUNT(*)
FROM (
    SELECT a.codePays
    FROM athlete a
    NATURAL JOIN rangIndividuel r
    WHERE r.rang <= 3

    UNION ALL

    SELECT a.codePays
    FROM athlete a
    NATURAL JOIN athletesEquipe aq
    NATURAL JOIN rangEquipe rq
    WHERE rq.rang <= 3
) AS medals
GROUP BY codePays;

"""

db.execute(query).df()

,codePays,count_star()
0,RUS,63
1,NED,28
2,CZE,11
3,CAN,89
4,SWE,55
5,SUI,31
6,SLO,10
7,ITA,14
8,POL,7
9,KOR,13


#### **Q9**. 
Pour chaque pays, le nombre de médailles d'or gagnées en équipe ou en individuel (20 lignes).

In [69]:
query="""
select codepays , count(*)
from (
        select a.codepays
        from athlete a 
        natural join rangindividuel r
        where r.rang = 1
        
        union all

        select distinct a.codepays   
        from athlete a
        natural join athletesequipe aq 
        natural join rangequipe rq 
        where rq.rang = 1
        
    ) as medals

group by codepays
"""

db.execute(query).df()

,codePays,count_star()
0,RUS,8
1,NED,7
2,CZE,2
3,CAN,6
4,SUI,6
5,SLO,2
6,POL,4
7,KOR,3
8,USA,9
9,SWE,1


#### **Q10**. 
Pour chaque pays, le nombre de médailles d'or, le nombre de médaille d'argent et le nombre de médaille de bronze gagnées. Conseil : utilisez des sous-requêtes dans  SELECT (206 lignes).

In [ ]:
query="""

"""

db.execute(query).df()

Sous-requête dans FROM (on n'obtient que les pays avec au moins une médaille d'or, d'argent et de bronze): 17 lignes 

In [ ]:
query="""

"""

db.execute(query).df()

###  Requêtes top-k  

#### **Q11**. 
Les dix épreuves les plus longues.

In [ ]:
query="""

"""

db.execute(query).df()